## Module 6 Homework 

In this homework, we're going to extend Module 5 Homework and learn about streaming with PySpark.

Instead of Kafka, we will use Red Panda, which is a drop-in
replacement for Kafka. 

Ensure you have the following set up (if you had done the previous homework and the module):

- Docker (see [module 1](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/01-docker-terraform))
- PySpark (see [module 5](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/05-batch/setup))

For this homework we will be using the files from Module 5 homework:

- Green 2019-10 data from [here](https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz)


In [1]:
import json
import time
import pandas as pd
import datetime

t0 = time.time()
from kafka import KafkaProducer
def json_serializer(data):
    return json.dumps(data).encode("utf-8")
server = "localhost:9092"

producer = KafkaProducer(bootstrap_servers=[server], value_serializer=json_serializer)
producer.bootstrap_connected()
t1 = time.time()

In [2]:
t1-t0

0.5796778202056885

## Question 1: Redpanda version

In [3]:
!docker exec -t redpanda-1 rpk topic delete green-topic

TOPIC        STATUS
green-topic  UNKNOWN_TOPIC_OR_PARTITION: This server does not host this topic-partition.


In [41]:
!docker exec -it redpanda-1 rpk version

Version:     v24.2.18
Git ref:     f9a22d4430
Build date:  2025-02-14T12:52:55Z
OS/Arch:     linux/amd64
Go version:  go1.23.1

Redpanda Cluster
  node-1  v24.2.18 - f9a22d443087b824803638623d6b7492ec8221f9


## Question 2. Creating a topic

In [4]:
!docker exec -it redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  OK


In [5]:
!docker exec -it redpanda-1 rpk topic list

NAME         PARTITIONS  REPLICAS
green-trips  1           1


In [8]:
!docker exec -it redpanda-1 rpk topic consume green-trips

^C


## Question 3. Connecting to the Kafka server

In [11]:
def json_serializer(data):
    return json.dumps(data, default=str).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

## Question 4. Sending data to the stream

In [16]:
# Загрузим данные
path_data ='https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz'
df_green = pd.read_csv(path_data, compression="gzip", low_memory=False)
# Выведем первые 5 строк для проверки
df_green.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2.0,2019-10-01 00:26:02,2019-10-01 00:39:58,N,1.0,112,196,1.0,5.88,18.0,0.50,0.5,0.00,0.0,NaN,0.3,19.30,2.0,1.0,0.0
1,1.0,2019-10-01 00:18:11,2019-10-01 00:22:38,N,1.0,43,263,1.0,0.80,5.0,3.25,0.5,0.00,0.0,NaN,0.3,9.05,2.0,1.0,0.0
2,1.0,2019-10-01 00:09:31,2019-10-01 00:24:47,N,1.0,255,228,2.0,7.50,21.5,0.50,0.5,0.00,0.0,NaN,0.3,22.80,2.0,1.0,0.0
3,1.0,2019-10-01 00:37:40,2019-10-01 00:41:49,N,1.0,181,181,1.0,0.90,5.5,0.50,0.5,0.00,0.0,NaN,0.3,6.80,2.0,1.0,0.0
4,2.0,2019-10-01 00:08:13,2019-10-01 00:17:56,N,1.0,97,188,1.0,2.52,10.0,0.50,0.5,2.26,0.0,NaN,0.3,13.56,1.0,1.0,0.0


In [17]:
columns_needed = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "tip_amount",
]

df_green = df_green[columns_needed]

df_green.info()  # Проверим структуру

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476386 entries, 0 to 476385
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   lpep_pickup_datetime   476386 non-null  object 
 1   lpep_dropoff_datetime  476386 non-null  object 
 2   PULocationID           476386 non-null  int64  
 3   DOLocationID           476386 non-null  int64  
 4   passenger_count        387007 non-null  float64
 5   trip_distance          476386 non-null  float64
 6   tip_amount             476386 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 25.4+ MB


In [53]:
df_green.to_csv('data/green_tripdata_2019-10.csv', index=False)

In [7]:
df_green=pd.read_csv('data/green_tripdata_2019-10.csv')

In [9]:
df_green.info()  # Проверим структуру

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476386 entries, 0 to 476385
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   lpep_pickup_datetime   476386 non-null  datetime64[ns]
 1   lpep_dropoff_datetime  476386 non-null  datetime64[ns]
 2   PULocationID           476386 non-null  int64         
 3   DOLocationID           476386 non-null  int64         
 4   passenger_count        387007 non-null  float64       
 5   trip_distance          476386 non-null  float64       
 6   tip_amount             476386 non-null  float64       
dtypes: datetime64[ns](2), float64(3), int64(2)
memory usage: 25.4 MB


In [12]:
topic_name = 'green-trips'
t0 = time.time()
for row in df_green.itertuples(index=False):
    row_dict = {col: getattr(row, col) for col in row._fields}
    # row_dict['event_timestamp'] = time.time() * 1000
    producer.send(topic_name, value=row_dict)

producer.flush()

t1 = time.time()

print(f'It took {round(t1 - t0, 2)} seconds to send these messages.')

producer.close()

It took 76.78 seconds to send these messages.


In [97]:
(t0, t1, t1-t0)

(1741532043.423154, 1741532153.4129815, 109.98982739448547)

In [13]:
row_count = df_green.shape[0]
print("Number of rows:", row_count)

Number of rows: 476386


In [33]:
!docker-compose exec jobmanager ./bin/flink run -py /opt/src/job/taxi_job.py --pyFiles /opt/src

Job has been submitted with JobID 748767098af829e891316825482f7cdf
^C
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/flink/opt/python/py4j-0.10.9.3-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/flink/opt/python/py4j-0.10.9.3-src.zip/py4j/java_gateway.py", line 1217, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/local/lib/python3.7/socket.py", line 589, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
Traceback (most recent call last):
  File "/usr/local/lib/python3.7/runpy.py", line 193, in _run_module_as_main
    "__main__", mod_spec)
  File "/usr/local/lib/python3.7/runpy.py", line 85, in _run_code
    exec(code, run_globals)
  File "/tmp/pyflink/b0805ac8-3db4-4809-84db-a6d63eef3da5/61a5e3ce-9b31-4c59-95f6-f75fc03468cb/taxi_job.py", line 89, in <module>
    log_processing()
  File "/tmp/pyflink/b0805ac8-3db

In [44]:
!docker-compose exec jobmanager ./bin/flink run -py /opt/src/job/session_job.py --pyFiles /opt/src

Job has been submitted with JobID 1b1094252d1c3ef3b4e38bf3a61e29fe
^C
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/flink/opt/python/py4j-0.10.9.3-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/flink/opt/python/py4j-0.10.9.3-src.zip/py4j/java_gateway.py", line 1217, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/local/lib/python3.7/socket.py", line 589, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt

------------------------------------------------------------
 The program finished with the following exception:

org.apache.flink.client.program.ProgramInvocationException: The main method caused an error: Shutdown in progress
	at org.apache.flink.client.program.PackagedProgram.callMainMethod(PackagedProgram.java:372)
	at org.apache.flink.client.program.PackagedProgram.invokeInteractiveModeForExecution(Packaged

In [ ]:
import pyspark
from pyspark.sql import SparkSession

pyspark_version = pyspark.__version__
kafka_jar_package = f"org.apache.spark:spark-sql-kafka-0-10_2.12:{pyspark_version}"

spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("GreenTripsConsumer") \
    .config("spark.jars.packages", kafka_jar_package) \
    .getOrCreate()

In [4]:
green_stream = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "green-trips") \
    .option("startingOffsets", "earliest") \
    .load()

In [14]:
def peek(mini_batch, batch_id):
    first_row = mini_batch.take(1)

    if first_row:
        print(first_row[0])

query = green_stream.writeStream.foreachBatch(peek).start()

24/03/16 06:38:45 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1987d72d-ba6c-438b-ab64-1e9b46530553. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/03/16 06:38:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
24/03/16 06:38:47 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [15]:
query.stop()


24/03/16 06:38:51 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (de-zoomcamp.europe-west1-b.c.ny-rides-marfanyan.internal executor driver): TaskKilled (Stage cancelled: Job 0 cancelled part of cancelled job group 5d86e8fa-c511-4e78-92a9-8810b3f9f9dc)


## Question 7: Most popular destination

In [ ]:
green_stream = green_stream.withColumn("timestamp", F.current_timestamp())




popular_destinations = green_stream.groupBy(
    F.window(
        green_stream.timestamp,
        windowDuration='5 minutes'
    ),
    green_stream.DOLocationID,
).count().orderBy(F.desc('count'))

In [ ]:
query = popular_destinations \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .start()

query.awaitTermination()
query.stop()

24/03/16 06:56:36 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9ba80686-f698-4e57-85f0-eeada4c37eb2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/03/16 06:56:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
24/03/16 06:56:37 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+------------+-----+
|window|DOLocationID|count|
+------+------------+-----+
+------+------------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+------------+-----+
|window                                    |DOLocationID|count|
+------------------------------------------+------------+-----+
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|74          |7564 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|42          |6708 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|41          |6148 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|75          |5343 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|7           |5065 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|129         |4990 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|166         |4838 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|236         |3422 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|238         |3197 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|223         |3197 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|82          |3122 |
|{2024-

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+------------+-----+
|window                                    |DOLocationID|count|
+------------------------------------------+------------+-----+
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|74          |16717|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|42          |14671|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|41          |13374|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|75          |11870|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|129         |11297|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|7           |10932|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|166         |10503|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|236         |7584 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|223         |7260 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|238         |7001 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|82          |6706 |
|{2024-

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+------------+-----+
|window                                    |DOLocationID|count|
+------------------------------------------+------------+-----+
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|74          |17270|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|42          |15353|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|41          |13754|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|75          |12428|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|129         |11671|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|7           |11277|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|166         |10668|
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|236         |7782 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|223         |7436 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|238         |7184 |
|{2024-03-16 06:55:00, 2024-03-16 07:00:00}|82          |7021 |
|{2024-